# 4.1 Setting up a quantum dynamics calculation on the IonQ platform with Qiskit

Versi ini mengganti semua bagian IBM Quantum menjadi workflow IonQ. Default backend adalah `simulator`; untuk hardware, ganti ke backend QPU yang kamu akses di IonQ Cloud.

In [ ]:
# ============================================================
# Script 4.1 (IonQ version): Package imports untuk Qiskit 2.x
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as LA

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Operator
from qiskit.circuit.library import UnitaryGate

from qiskit_ionq import IonQProvider

In [ ]:
# ============================================================
# Script 4.2: Hamiltonian dan Propagator
# ============================================================

J  = 1
h0 = -0.5
h1 =  0.5

X = np.array([[0, 1 ], [1,  0]], dtype=complex)
Y = np.array([[0,-1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0 ], [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

H = (0.5 * (h0 * np.kron(Z, I) + h1 * np.kron(I, Z))
     + (J / 4) * (np.kron(X, X) + np.kron(Y, Y) + np.kron(Z, Z)))

U = LA.expm(-1j * H)

print("Hamiltonian H (4×4):")
print(np.round(H, 3))
print("\nPropagator U = exp(-iH):")
print(np.round(U, 3))

assert np.allclose(U.conj().T @ U, np.eye(4), atol=1e-10), "U is not unitary!"
print("\nVerifikasi: U adalah unitary ✓")

In [ ]:
# ============================================================
# Script 4.3: Classical propagation (benchmark)
# ============================================================

psi_init = np.array([1, 0, 0, 0], dtype=complex)
psi_fin = U @ psi_init

print("State awal  |ψ(0)⟩:", np.round(psi_init, 4))
print("State akhir |ψ(τ)⟩:", np.round(psi_fin, 4))
print("Probabilitas klasik:", np.round(np.abs(psi_fin)**2, 4))

In [ ]:
# ============================================================
# Script 4.4–4.6: Quantum Circuit Setup untuk IonQ
# ============================================================

qreg = QuantumRegister(2, name='q')
creg = ClassicalRegister(2, name='c')
entangler = QuantumCircuit(qreg, creg, name="2-site-Heisenberg-IonQ")

# Gunakan UnitaryGate agar bisa ditranspile ke gate yang didukung IonQ
U_gate = UnitaryGate(U, label="U")

entangler.append(U_gate, [0, 1])
entangler.measure(0, 0)
entangler.measure(1, 1)

print(entangler.draw(output='text'))

In [ ]:
# ============================================================
# Script 4.7–4.9: IonQ Simulator / QPU
# ============================================================

# Pilihan backend:
# - "simulator" untuk simulator ideal IonQ
# - "qpu.forte-1" atau backend QPU lain yang kamu punya aksesnya
backend_name = "simulator"

# Jika mau pakai hardware IonQ, ganti jadi:
# backend_name = "qpu.forte-1"

# Masukkan API key IonQ di sini, atau set environment variable dan pakai IonQProvider()
IONQ_API_KEY = os.getenv("IONQ_API_KEY")  # boleh None jika provider bisa menemukan credential
provider = IonQProvider(IONQ_API_KEY) if IONQ_API_KEY else IonQProvider()

backend = provider.get_backend(backend_name)
print("Backend IonQ:", backend)

# Transpile circuit ke basis gate yang diterima backend IonQ
# optimization_level=1 direkomendasikan untuk IonQ Qiskit workflow
qc_ionq = transpile(entangler, backend=backend, optimization_level=1)

print("\nTranspiled circuit:")
print(qc_ionq.draw(output='text'))

job = backend.run(qc_ionq, shots=2000)
print("\nJob ID:", job.job_id())

result = job.result()
counts_ionq = result.get_counts()

print("\nCounts IonQ:", counts_ionq)

In [ ]:
# ============================================================
# Script 4.10: Visualisasi perbandingan klasik vs IonQ
# ============================================================

classical_probs = np.abs(psi_fin)**2

labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
keys_bin = ['00', '01', '10', '11']

def get_prob(probs_dict, key):
    return probs_dict.get(key, 0)

shots = 2000
ionq_probs = {k: v / shots for k, v in counts_ionq.items()}
ionq_arr = [get_prob(ionq_probs, k) for k in keys_bin]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(4)
ax.bar(x - 0.2, classical_probs, 0.4, label='Classical (exact)')
ax.bar(x + 0.2, ionq_arr,       0.4, label=f'IonQ {backend_name}')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Probability')
ax.set_ylim(0, 1)
ax.set_title('2-site Heisenberg: Classical vs IonQ')
ax.legend()
plt.tight_layout()
plt.show()

print("Probabilitas klasik:", {k: round(v, 4) for k, v in zip(keys_bin, classical_probs)})
print("Probabilitas IonQ:", {k: round(v, 4) for k, v in ionq_probs.items()})

## Catatan

Jika backend `qpu.forte-1` tidak tersedia untuk akunmu, gunakan dulu `simulator`. Untuk hardware, backend yang bisa diakses bergantung pada izin di IonQ Cloud.